# 01 — Data Load & Merge

**Data source:** [IEEE-CIS Fraud Detection](https://www.kaggle.com/c/ieee-fraud-detection) — Kaggle competition dataset provided by Vesta Corporation (e-commerce, card-not-present transactions).

Raw files (`train_transaction.csv`, `train_identity.csv`) are **not committed** to this repo — download them from Kaggle into `data/raw/`.

**Time note:** `TransactionDT` is a relative timedelta in seconds from an unstated reference date, not a real timestamp. Dates render as 1970 after conversion — cosmetic only; hour-of-day and all relative time windows remain valid.

In [ ]:
# CELL 1 - Load raw IEEE-CIS files + match-rate sanity check
import pandas as pd

trans = pd.read_csv("../data/raw/train_transaction.csv")
identity = pd.read_csv("../data/raw/train_identity.csv")

print(f"transactions: {trans.shape}")
print(f"identity:     {identity.shape}")

match_rate = trans["TransactionID"].isin(identity["TransactionID"]).mean()
print(f"transactions with identity data: {match_rate:.1%}")  # expect ~24%

In [ ]:
# CELL 2 - Merge transaction + identity
df = trans.merge(identity, on="TransactionID", how="left")
print(f"merged: {df.shape}")
assert len(df) == len(trans), "Left join changed row count!"

In [ ]:
# CELL 3 - Convert time + sort chronologically
# TransactionDT is seconds from an UNSTATED reference date (per the
# competition), not a real timestamp. With the default epoch, dates
# render as 1970 - cosmetic only. Hour-of-day and all relative
# windows stay valid because day boundaries align every 86,400 s.
df["TransactionDT"] = pd.to_datetime(df["TransactionDT"], unit="s")
df = df.sort_values("TransactionDT").reset_index(drop=True)

In [ ]:
# CELL 4 - Synthetic customer ID (with NaN handling)
# card2 is missing for ~1-2% of rows (visible in head() output).
# Fill BEFORE building the ID so those customers are kept, not dropped.
n_missing = df["card2"].isna().sum()
print(f"card2 missing: {n_missing:,} rows ({n_missing / len(df):.2%}) -> filled with -1")

df["card2"] = df["card2"].fillna(-1)
df["customer_id"] = df["card1"].astype(str) + "_" + df["card2"].astype(str)
print(f"unique synthetic customers: {df['customer_id'].nunique():,}")

In [ ]:
# CELL 5 - First fraud-over-time check
import matplotlib.pyplot as plt

ax = (
    df.set_index("TransactionDT")["isFraud"]
      .resample("D").mean()
      .plot(title="Fraud Rate Over Time (days from unknown reference)")
)
ax.set_ylabel("Daily fraud rate")
plt.show()

In [ ]:
# CELL 6 - Select core columns (explicit about what gets dropped)
core_cols = [
    "TransactionDT", "TransactionAmt", "customer_id",
    "card1", "card2", "ProductCD", "isFraud",
]
df_core = df[core_cols].copy()

missing = df_core.isna().sum()
print("Remaining NaNs:", dict(missing[missing > 0]) or "none")

df_core = df_core.dropna()  # safety net; removes ~0 rows after the fill
print(f"rows kept: {len(df_core):,} / {len(df):,}")

In [ ]:
# CELL 7 - Save clean base dataset
df_core.to_csv("../data/processed/base_transactions.csv", index=False)
print(f"Saved base_transactions.csv: {df_core.shape}")
print(f"Overall fraud rate: {df_core['isFraud'].mean():.2%}")  # expect ~3.5%